# SAM-WM frozen Kaggle research benchmark

Run top-to-bottom. The repository code, benchmark loaders, pre-freeze baseline/ablation suite,
held-out gates, two zero-shot OOD evaluations, and evidence pipeline are already defined in Git.
Kaggle supplies the real dataset bytes, GPU execution, learned checkpoints, and measured results.

**Leakage boundary:** only Freiburg development/validation evidence may be inspected before the
freeze cell. Freiburg held-out and both OOD targets are opened only after the source/config/model
choice is frozen.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
import urllib.error
import urllib.request
from pathlib import Path

REPOSITORY = "AnnyaB/SAM-WM"
WORK_ROOT = Path("/kaggle/working")
REPO = WORK_ROOT / "SAM-WM"
SEEDS = (17, 29, 42, 73, 101)

def github_token() -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        value = None
    return value.strip() if isinstance(value, str) and value.strip() else None

def github_json(url: str, token: str | None) -> dict:
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "sam-wm-kaggle",
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"
    request = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            return json.load(response)
    except urllib.error.HTTPError as exc:
        if exc.code in {401, 403, 404}:
            raise RuntimeError(
                "GitHub source is not accessible. If the repository is private, "
                "add a Kaggle Secret named GITHUB_TOKEN with read access to AnnyaB/SAM-WM."
            ) from exc
        raise

def download_archive(url: str, dst: Path, token: str | None) -> None:
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "sam-wm-kaggle",
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"
    request = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(request, timeout=120) as response, dst.open("wb") as handle:
        shutil.copyfileobj(response, handle)

def safe_extract(archive: Path, dst: Path) -> Path:
    dst.mkdir(parents=True, exist_ok=True)
    root = dst.resolve()
    with tarfile.open(archive, mode="r:gz") as tf:
        members = tf.getmembers()
        for member in members:
            target = (root / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f"unsafe archive path: {member.name}")
            if member.issym() or member.islnk():
                raise RuntimeError(f"archive links are not accepted: {member.name}")
        tf.extractall(root)
    directories = [path for path in root.iterdir() if path.is_dir()]
    if len(directories) != 1:
        raise RuntimeError(f"unexpected GitHub archive layout: {directories}")
    return directories[0]

token = github_token()
meta = github_json(f"https://api.github.com/repos/{REPOSITORY}/commits/main", token)
SOURCE_SHA = str(meta["sha"])
if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_SHA):
    raise RuntimeError("GitHub returned an invalid commit SHA")

if REPO.exists():
    shutil.rmtree(REPO)

with tempfile.TemporaryDirectory(prefix="samwm-source-") as tmp_name:
    tmp = Path(tmp_name)
    archive = tmp / "source.tar.gz"
    download_archive(
        f"https://api.github.com/repos/{REPOSITORY}/tarball/{SOURCE_SHA}",
        archive,
        token,
    )
    extracted = safe_extract(archive, tmp / "extract")
    shutil.copytree(extracted, REPO)

os.chdir(REPO)
(REPO / "artifacts").mkdir(exist_ok=True)
(REPO / "artifacts" / "FROZEN_SOURCE_SHA.txt").write_text(
    SOURCE_SHA + "\n", encoding="utf-8"
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"],
    check=True,
)
subprocess.run(["make", f"PYTHON={sys.executable}", "verify"], check=True)
print("Frozen source SHA:", SOURCE_SHA)


In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training.")
print("GPU:", torch.cuda.get_device_name(0))


## Pre-freeze research suite — validation only

This is the experiment code that must exist **before** any held-out label is touched.
It runs the exact full SAM-WM branch, three non-trainable sanity baselines, seven retrained
structural ablations, and two objective controls across the frozen five seeds.

This stage is intentionally forbidden from reading Freiburg held-out, Novi Sad targets, or
FAIRUrbTemp targets.


In [ ]:
def run_python(*args: str) -> None:
    subprocess.run([sys.executable, *args], check=True)

run_python(
    "research.py",
    "--stage", "all-pre-freeze",
    "--out", "artifacts/research",
)

manifest = Path("artifacts/research/PRE_FREEZE_MANIFEST.json")
print(manifest.read_text(encoding="utf-8"))


## Inspect Freiburg validation evidence only

At this point you may inspect validation results and decide whether this exact research version
is scientifically usable. If you change architecture, loss, preprocessing, QC, splits, or
hyperparameters, start a new run and do **not** open held-out/OOD results from the discarded run.


In [ ]:
validation_rows = []
for seed in SEEDS:
    path = Path(f"artifacts/research/full/seed_{seed}/validation_metrics.json")
    payload = json.loads(path.read_text(encoding="utf-8"))
    validation_rows.append(
        {
            "seed": seed,
            "mae": payload["validation"]["mae"],
            "rmse": payload["validation"]["rmse"],
            "bias": payload["validation"]["bias"],
        }
    )
validation_rows


## Freeze the reported full SAM-WM branch

Run this only after deciding that no more train/validation-driven changes will be made.
The freeze hashes the exact source, config, five full checkpoints, and the complete pre-freeze
research evidence bundle.


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

pre_freeze = REPO / "artifacts/research/PRE_FREEZE_MANIFEST.json"
if not pre_freeze.exists():
    raise RuntimeError("Pre-freeze research manifest is missing.")

checkpoints = {
    f"seed_{seed}": REPO / f"artifacts/research/full/seed_{seed}/best.pt"
    for seed in SEEDS
}
missing = [str(path) for path in checkpoints.values() if not path.exists()]
if missing:
    raise RuntimeError(f"Full SAM-WM checkpoints are incomplete: {missing}")

freeze = {
    "protocol": "SAM_WM_FINAL_FREEZE_V1",
    "source_sha": (REPO / "artifacts/FROZEN_SOURCE_SHA.txt").read_text(
        encoding="utf-8"
    ).strip(),
    "config_sha256": sha256_file(REPO / "config/train.yaml"),
    "pre_freeze_manifest_sha256": sha256_file(pre_freeze),
    "seeds": list(SEEDS),
    "full_checkpoints": {
        name: sha256_file(path) for name, path in checkpoints.items()
    },
    "rule": (
        "No architecture, objective, preprocessing, QC, split, or hyperparameter change "
        "is allowed after this manifest for the reported held-out/OOD run."
    ),
}
freeze_path = REPO / "artifacts/FREEZE_MANIFEST.json"
freeze_path.write_text(
    json.dumps(freeze, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(freeze_path.read_text(encoding="utf-8"))


## Freiburg final test — open once after freeze


In [ ]:
for seed in SEEDS:
    run_python(
        "eval.py",
        "--checkpoint", f"artifacts/research/full/seed_{seed}/best.pt",
        "--data", "freiburg",
        "--split", "heldout",
        "--open-heldout",
        "--out", f"artifacts/eval/seed_{seed}",
    )


## OOD-1 — Novi Sad zero-shot

No fine-tuning, no OOD-label recalibration, and no architecture selection from Novi Sad.


In [ ]:
for seed in SEEDS:
    run_python(
        "eval.py",
        "--checkpoint", f"artifacts/research/full/seed_{seed}/best.pt",
        "--data", "novisad",
        "--split", "heldout",
        "--open-heldout",
        "--out", f"artifacts/eval/seed_{seed}",
    )


## OOD-2 — FAIRUrbTemp unseen-city zero-shot

Attach the official DOI `10.48620/93247` extracted dataset as a Kaggle Input. Choose
`FAIR_CITY` using metadata/coverage criteria **before viewing any SAM-WM result** and keep
that city identical for all five seeds.


In [ ]:
FAIR_ROOT = Path("/kaggle/input/REPLACE_WITH_FAIRURBTEMP/extracted-root")
FAIR_CITY = "REPLACE_WITH_PREREGISTERED_CITY"

if "REPLACE_WITH" in str(FAIR_ROOT) or FAIR_CITY.startswith("REPLACE_WITH"):
    raise RuntimeError(
        "Set FAIR_ROOT and FAIR_CITY before opening FAIRUrbTemp held-out evaluation."
    )
if not FAIR_ROOT.exists():
    raise FileNotFoundError(FAIR_ROOT)

for seed in SEEDS:
    run_python(
        "eval.py",
        "--checkpoint", f"artifacts/research/full/seed_{seed}/best.pt",
        "--data", "fairurbtemp",
        "--root", str(FAIR_ROOT),
        "--city", FAIR_CITY,
        "--split", "heldout",
        "--open-heldout",
        "--out", f"artifacts/eval/seed_{seed}",
    )


## Aggregate immutable evidence and generate figures


In [ ]:
run_python(
    "summarize.py",
    "--root", "artifacts/eval",
    "--out", "artifacts/summary.json",
)

for seed in SEEDS:
    run_python(
        "plot.py",
        f"artifacts/eval/seed_{seed}/freiburg_heldout_metrics.json",
        f"artifacts/eval/seed_{seed}/novisad_heldout_metrics.json",
        f"artifacts/eval/seed_{seed}/fairurbtemp_heldout_metrics.json",
        "--out", f"artifacts/figures/seed_{seed}",
    )

print(Path("artifacts/summary.json").read_text(encoding="utf-8"))
